In [5]:
"""Fine-tune Qwen/Qwen3.5-4B on Kyudan/MathBridge with QLoRA (4-bit).

This script is designed for constrained GPUs such as RTX 3070 8GB VRAM.
"""

from __future__ import annotations

import argparse
import inspect
import os
from typing import Any

import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    set_seed,
)
from trl import SFTConfig, SFTTrainer


SYSTEM_PROMPT = (
    "Tu es un assistant mathematique qui traduit de l'anglais parle vers des equations."
)

ModuleNotFoundError: Could not import module 'BloomPreTrainedModel'. Are this object's requirements defined correctly?

In [ ]:
def main() -> None:
    args = parse_args()
    set_seed(args.seed)

    if not torch.cuda.is_available():
        raise EnvironmentError(
            "CUDA GPU is required for this script. "
        )

    hf_token = args.hf_token or os.getenv("HF_TOKEN")

    try:
        # Step 5: load quantized model + LoRA adapter.
        model, tokenizer = build_model_and_tokenizer(
            model_name=args.model_name,
            lora_r=args.lora_r,
            lora_alpha=args.lora_alpha,
            lora_dropout=args.lora_dropout,
            hf_token=hf_token,
        )

        # Step 6: build the training text field from dataset columns.
        train_dataset, test_dataset = load_and_prepare_dataset(
            dataset_name=args.dataset_name,
            dataset_split=args.dataset_split,
            tokenizer=tokenizer,
            hf_token=hf_token,
            dataset_fraction=args.dataset_fraction,
            max_train_samples=args.max_train_samples,
            sample_seed=args.seed,
        )

        # Step 7: configure low-memory SFT training.
        sft_kwargs = {
            "output_dir": args.output_dir,
            "per_device_train_batch_size": args.per_device_train_batch_size,
            "gradient_accumulation_steps": args.gradient_accumulation_steps,
            "learning_rate": args.learning_rate,
            "max_steps": args.max_steps,
            "warmup_ratio": args.warmup_ratio,
            "logging_steps": args.logging_steps,
            "save_steps": args.save_steps,
            "save_total_limit": 2,
            "fp16": False,
            "bf16": True,
            "gradient_checkpointing": True,
            "optim": "paged_adamw_8bit",
            "lr_scheduler_type": "cosine",
            "report_to": "none",
            "remove_unused_columns": False,
            "dataloader_pin_memory": False,
            "packing": False,
        }

        sft_signature = inspect.signature(SFTConfig.__init__).parameters
        if "max_seq_length" in sft_signature:
            sft_kwargs["max_seq_length"] = args.max_seq_length
        elif "max_length" in sft_signature:
            sft_kwargs["max_length"] = args.max_seq_length

        if "gradient_checkpointing_kwargs" in sft_signature:
            sft_kwargs["gradient_checkpointing_kwargs"] = {"use_reentrant": False}

        if "dataset_text_field" in sft_signature:
            sft_kwargs["dataset_text_field"] = "text"

        sft_config = SFTConfig(**sft_kwargs)

        trainer_kwargs = {
            "model": model,
            "args": sft_config,
            "train_dataset": train_dataset,
        }

        # Keep compatibility between TRL versions where tokenizer arg was renamed.
        signature_params = inspect.signature(SFTTrainer.__init__).parameters
        if "dataset_text_field" in signature_params:
            trainer_kwargs["dataset_text_field"] = "text"
        if "processing_class" in signature_params:
            trainer_kwargs["processing_class"] = tokenizer
        else:
            trainer_kwargs["tokenizer"] = tokenizer

        # Step 8: train and export the final LoRA adapter.
        trainer = SFTTrainer(**trainer_kwargs)
        torch.cuda.empty_cache()
        trainer.train()

        final_dir = os.path.join(args.output_dir, "final")
        trainer.model.save_pretrained(final_dir)
        tokenizer.save_pretrained(final_dir)
        print(f"Training complete. Adapter and tokenizer saved to: {final_dir}")

    except OSError as err:
        raise RuntimeError(
            "Failed to load model or tokenizer. Ensure model ID is valid: "
            f"{args.model_name}."
        ) from err
    except RuntimeError as err:
        if "out of memory" in str(err).lower():
            raise RuntimeError(
                "CUDA OOM detected. Try lowering --max_seq_length to 128, "
                "or reduce --gradient_accumulation_steps and --save_steps frequency."
            ) from err
        raise


if __name__ == "__main__":
    main()
